In [1]:
%load_ext autoreload
%autoreload 2

In [106]:
## going to test the new setup
from nba import NBAbase, NBAetl, NBAmodels, NBAdata
#from dev import NBAdata
from betting import odds
import pandas as pd
import seaborn as sns
import sqlite3
import shutil
import numpy as np
from betting import funcs
odds = funcs.odds()
import matplotlib.pyplot as plt

In [10]:
q = '''CREATE TABLE IF NOT EXISTS predictions (
    player_id       TEXT,
    date            TEXT,
    over_under      TEXT,
    number          REAL,
    model_prob      REAL,
    FanDuels         REAL,
    DraftKings      REAL,
    theScore_Bet         REAL,
    ev_fanduel      REAL,
    ev_draftkings   REAL,
    ev_espnbet      REAL,
    bet_book        TEXT,
    final_line      REAL,
    bet_amount      REAL,
    result          REAL,
    PRIMARY KEY (player_id, date, over_under, number)
);'''

In [7]:
etl = NBAetl.etl()
data = NBAdata.data()
odds = funcs.odds()

In [12]:
etl.cur.execute(q)
etl.conn.commit()

In [15]:
#run file
#get yesterday's data and load
yst = (dt.datetime.today() + pd.to_timedelta(-1,unit='day')).strftime(format='%Y-%m-%d')
gids = etl.get_games(yst,yst)
print('Updating for {}'.format(yst))
etl.update_player_log([yst])
time.sleep(np.random.randint(5,15))
try:
    etl.update_shots_allowed([yst])
except:
    print('shots not updated yet')
time.sleep(np.random.randint(5,15)) 
etl.update_teamLog(gids.GAME_ID.unique())

#get the model and run the data through the pipeline
threes = NBAmodels.models('threes')

td = data.threes_pipe(threes.data)

td = data.clean_na(td)
td = td[td.game_date==(dt.datetime.today()).strftime('%Y-%m-%d')]
td = threes.standRobust_scaler(td)
#get predictions
preds= threes.model.predict(sm.add_constant(td.filter(threes.features),has_constant='add'))

#name alterations
idInfo = threes.data[threes.data.game_date==dt.datetime.today().strftime('%Y-%m-%d')][['name','team','game_id']]
idInfo.name = data.standardize_names(idInfo.name)

#odds pull
overs = odds.oddsTable(preds,idInfo)


#%2cespnbet
def fetch_odds(market):
    url = odds.market_vars.get(market).get('url')
    df = pd.DataFrame()
    events,akey = odds.oddsData(odds.nbaEvents,usePaid=True)
    for event in events:
        r = requests.get(url.format(event,akey))
        game = r.json()
        for key in game.get('bookmakers'):
            bk = key.get('title')
            for mrkt in key.get('markets'):
                temp = pd.DataFrame(mrkt.get('outcomes'))
                temp.columns = ['over_under','name','price',odds.market_vars.get(market).get('col_name')]
                temp['book'] = bk
                df = pd.concat([temp,df])   
    odf = df.pivot_table(index=['name',odds.market_vars.get(market).get('col_name'),'over_under'],columns=['book']).reset_index()
    odf.columns = [col[1] if col[1]!= '' else col[0] for col in odf.columns]
    return odf
#odds name updates
odf.name = data.standardize_names(odf.name)

#create the data table view
def bet_table(overs,odf,val_col):
    
    final = odf.merge(overs,how='left',on=['name',val_col,'over_under'])
    final['prob'] = np.where(final.value<0, round(abs(final.value) / (abs(final.value) + 100),4), round(100/(final.value +100),4))
    final['DKEV'] = ['{:.2%}'.format(odds.ev(p, odd)) for p,odd in zip(final.prob,final.DraftKings.replace(0,1))]
    final['FanDuelEV'] = ['{:.2%}'.format(odds.ev(p, odd))  for p,odd in zip(final.prob,final.FanDuel.replace(0,1))]
    #final['BetMGMEV'] = ['{:.2%}'.format(odds.ev(p, odd))  for p,odd in zip(final.prob,final.BetMGM)]
    final['espnEV'] = ['{:.2%}'.format(odds.ev(p, odd))  for p,odd in zip(final.prob,final['theScore Bet'])]
    #first row needs to be the tiebreaker row
    final['FanDuelAmount'] = [round(x * odds.budget * odds.kellyVal,2) for x in final.FanDuelKelly.values]
    final['DraftKingsAmount'] = [round(x * odds.budget * odds.kellyVal,2) for x in final.DKKelly.values]
    #final['BetMGMAmount'] = [round(x * odds.budget * odds.kellyVal,2) for x in final.BetMGMKelly.values]
    final['theScore BetAmount'] = [round(x * odds.budget * odds.kellyVal,2) for x in final.espnKelly]
    return final


NameError: name 'dt' is not defined

In [52]:
# run_daily.py

import datetime as dt
import time
import statsmodels.api as sm

def data_pull(run_date=None):
    run_date = run_date or (dt.datetime.today() + pd.to_timedelta(-1, unit='day')).strftime(format='%Y-%m-%d'))
    '''
    Our Daily run to pull in all game data for date requested.  Typically the prior day but can be adjusted for missing data.
    this is for a single day update, run individual functions for bulk loading
    inputs: Optional date, defaults to yesterday
    Outputs: None; provides print statements with updates, logging on errors with tracking data and will refresh opponent data table.
    '''
    gids = etl.get_games(run_date,run_date)
    print('Updating for {}'.format(run_date))
    etl.update_player_log([run_date])
    time.sleep(np.random.randint(5,15))
    etl.update_shots_allowed([run_date])
    time.sleep(np.random.randint(5,15)) 
    etl.update_teamLog(gids.GAME_ID.unique())
    data.refresh_opp_data()

def run_model(model_name,date=None):
    '''
    Run function for the model, this will get the model and produce the predictions both in a long and wide format
    inputs: model name, date that defaults to today
    ouput: long and wide dataframes of predicitons for the day
    '''
    date =  date or (dt.datetime.today() + pd.to_timedelta(-1, unit='day')).strftime(format='%Y-%m-%d')
    model = NBAmodels.models(model_name)
    pipe = model.get(pipe)
    td = pipe(model.data)
    td = data.clean_na(td)
    td = td[td.game_date == date]
    td = model.standRobust_scaler(td)
    preds = model.model.predict(sm.add_constant(td.filter(model.features), has_constant='add'))
    idInfo = model.data[model.data.game_date == date][['name','team','game_id']]
    idInfo.name = data.standardize_names(idInfo.name)
    return odds.oddsTable(preds, idInfo,odds.market_vars.get(model_name).get('col_name')),idInfo,model

data_pull()
overs, idInfo = run_model('threes')
odf = odds.fetch_odds('threes')
final = odds.bet_table(overs,odf,'threesMade')


# def write_predictions(final, conn):
#     today = dt.datetime.today().strftime('%Y-%m-%d')
#     final['date'] = today
#     final['bet_book'] = None
#     final['final_line'] = None
#     final['bet_amount'] = None
#     final['result'] = None
#     final.rename(columns={'threesMade':'number','name':'player_id'}, inplace=True)
#     final[['player_id','date','over_under','number','prob',
#            'DraftKings','FanDuel','ESPN Bet',
#            'ev_draftkings','ev_fanduel','ev_espnbet',
#            'bet_book','final_line','bet_amount','result']].to_sql('predictions', conn, if_exists='append', index=False)



# if __name__ == '__main__':
#     yst = (dt.datetime.today() + pd.to_timedelta(-1, unit='day')).strftime('%Y-%m-%d')
#     conn = sqlite3.connect('nba.db')
#     run_data_update(yst)
#     overs = run_model()
#     odf, o = fetch_odds()
#     final = build_final(overs, odf, o)
#     write_predictions(final, conn)

[autoreload of betting.funcs failed: Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/opt/anaconda3/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.11/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 621, in _exec
  File "<frozen importlib._bootstrap_external>", line 940, in exec_module
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "/Users/ericcoxon/code/bets/betting/funcs.py", line 8, in <module>
    from constants import books
ModuleNotFoundError: No module named 'constants'
]


Free is out
{'Date': 'Sat, 11 Apr 2026 12:58:11 GMT', 'Content-Type': 'application/json; charset=utf-8', 'Content-Length': '3', 'Connection': 'keep-alive', 'X-Requests-Used': '141', 'X-Requests-Remaining': '19859', 'X-Requests-Last': '0', 'vary': 'Accept-Encoding', 'Apigw-Requestid': 'bp_Hqi58IAMEZiw='}


KeyError: 'name'

In [62]:
odds.market_vars.get('threes').get('col_name')

'threesMade'

In [31]:
end_date = (dt.datetime.today() + pd.to_timedelta(-2,unit='day')).strftime(format='%Y-%m-%d')
start_date = (dt.datetime.today() + pd.to_timedelta(-12,unit='day')).strftime(format='%Y-%m-%d')
df = pd.read_sql('''SELECT game_id,team_id,game_date,wide_fg2a,wide_fg3m,open_fg3m,open_fg3a
                      FROM shotsAllowed WHERE game_date BETWEEN '{}' AND '{}' '''.format(start_date,end_date),etl.conn)

In [121]:
idInfo = run.run_model(model_name,date)

In [122]:
idInfo['name'] = idInfo['name'].str.replace('.','')

In [92]:
date = '2026-03-26'
model_name='threes'
pipes = {'threes':data.threes_pipe}

In [87]:
t, idinfo = chk

In [276]:
overs,idInfo = run.run_model('threes',date)

In [277]:
#finding odds
odf = pd.read_csv('../nba/data/csv/{}odds.csv'.format(date))
odf.rename(columns = {'threesMade':'number'},inplace=True)
#overs.rename(columns = {'threesMade':'number',inplace=True)

In [278]:
final = odds.bet_table(overs,odf)

In [294]:
import re
amts = [col for col in final.columns if re.search('Amount$',col)!=None]

In [143]:
EVcols = [col for col in final.columns if col.find('Amount')>-1]

In [161]:
final[(final[EVcols]>1).sum(axis=1)>0][EVcols].max(axis=1)

2       1.55
7       4.01
16     24.78
17     36.04
18      2.94
       ...  
695    37.65
703    23.28
737     5.56
750    55.75
805    39.50
Length: 69, dtype: float64

In [175]:
','.join(pd.read_sql("select sql from sqlite_master where name = 'predictions' ",etl.conn).sql.values.tolist()).split('\n')

['CREATE TABLE predictions (',
 '    player_id       TEXT,',
 '    date            TEXT,',
 '    over_under      TEXT,',
 '    number          REAL,',
 '    model_prob      REAL,',
 '    FanDuels         REAL,',
 '    DraftKings      REAL,',
 '    theScore_Bet         REAL,',
 '    ev_fanduel      REAL,',
 '    ev_draftkings   REAL,',
 '    ev_espnbet      REAL,',
 '    bet_book        TEXT,',
 '    final_line      REAL,',
 '    bet_amount      REAL,',
 '    result          REAL, user TEXT,',
 '    PRIMARY KEY (player_id, date, over_under, number)',
 ')']

In [177]:
qbets = '''
CREATE TABLE bets (
     player_id       TEXT,
     date            TEXT,
     market          TEXT,
     over_under      TEXT,
     number          REAL,
     bet_book        TEXT,
     final_line      REAL,
     bet_amount      REAL,
     user TEXT,
     PRIMARY KEY (player_id, date, over_under, number)
     )
'''
qpreds = '''
CREATE TABLE predictions (
     player_id       TEXT,
     date            TEXT,
     market          TEXT,
     over_under      TEXT,
     number          REAL,
     model_prob      REAL,
     tracking_complete INTEGER,
     PRIMARY KEY (player_id, date, over_under, number)
     )
'''
qlines = '''
CREATE TABLE lines (
     player_id       TEXT,
     date            TEXT,
     market          TEXT,
     over_under      TEXT,
     number          REAL,
     FanDuel         REAL,
     DraftKings      REAL,
     theScore_Bet    REAL,
     PRIMARY KEY (player_id, date, over_under, number)
     )
'''

In [180]:
etl.cur.execute('DROP TABLE predictions')
etl.cur.execute(qpreds)
etl.cur.execute(qlines)
etl.conn.commit()

In [186]:
import glob
odds_files = glob.glob('../nba/data/csv/*odds.csv')

In [188]:
len(odds_files)

249

In [231]:
l = []
for f in odds_files:
    d = f[16:f.find('odds')]
    temp = pd.read_csv(f)
    temp['date'] = d
    temp.rename(columns = {'ESPN BET':'theScoreBet','theScore Bet':'theScoreBet'}
                ,inplace=True)
    
    

In [232]:
linesdf = pd.concat(l)

In [254]:
pd.read_sql('select * from lines',etl.conn)

,player_id,date,market,over_under,number,FanDuel,DraftKings,theScore_Bet
0,203932,2024-12-22,threes,Over,0.5,NaN,-191.5,-190.0
1,203932,2024-12-22,threes,Under,0.5,NaN,145.0,135.0
2,203932,2024-12-22,threes,Over,1.5,NaN,240.0,220.0
3,203932,2024-12-22,threes,Under,1.5,NaN,NaN,-350.0
4,203932,2024-12-22,threes,Over,2.5,NaN,800.0,525.0
...,...,...,...,...,...,...,...,...
141672,203897,2026-01-29,threes,Under,3.5,-330.0,NaN,-325.0
141673,203897,2026-01-29,threes,Over,4.5,550.0,541.0,500.0
141674,203897,2026-01-29,threes,Under,4.5,-1000.0,NaN,NaN
141675,203897,2026-01-29,threes,Over,5.5,1100.0,1120.0,1100.0


In [255]:
books = {
    'draftkings': {'col_prefix': 'DraftKings', 'odds_col': 'DraftKings'},
    'fanduel': {'col_prefix': 'FanDuel', 'odds_col': 'FanDuel'},
    'espnbet': {'col_prefix': 'espn', 'odds_col': 'theScore Bet'},
    'fanatcis': {'col_prefix': 'fanatics', 'odds_col': 'Fanatics'},
    'betmgm': {'col_prefix': 'BetMgm', 'odds_col': 'BetMGM'},
}